<!-- paired-role-banner -->
> 🟧 **실습 노트북 08 · 빈 코드 창**
>
> 오른쪽의 같은 이름 **정답 노트북**과 셀 위치를 맞춰 보세요. 먼저 정답을 가리고 직접
> 작성한 뒤 막힐 때 한 줄씩 비교하는 방식을 권장합니다. `NotImplementedError`가 있는 셀을
> 완성하면 됩니다.

# 08. CNN · RNN · Attention 비교 실습

순차 데이터를 처리하는 세 계열을 같은 관점으로 비교합니다. 먼저 작은 시계열의 다음 값을 예측하고, 이어서 짧은 한국어 문장을 분류합니다.

예상 시간: 100~140분 · 실행 장치: CUDA → MPS → CPU 자동 선택 · 외부 데이터 다운로드 없음

## 한눈에 보는 귀납적 편향

| 모델 | 정보를 섞는 방식 | 강점 | 주의점 |
|---|---|---|---|
| 1D CNN | 가까운 window를 병렬 convolution | 빠름, local pattern | 먼 관계에는 층/팽창 필요 |
| RNN/GRU | 이전 hidden state를 순서대로 갱신 | 순서 표현, streaming | 시간축 병렬화 제한 |
| Self-attention | 모든 token 쌍의 가중합 | 먼 관계, 병렬 학습 | 기본 비용이 $O(T^2)$ |

여기서는 GRU를 RNN 계열의 대표로 사용합니다. 어떤 모델이 항상 우월한 것이 아니라 데이터 크기, sequence 길이, latency 조건에 따라 선택합니다.

In [ ]:
from collections import Counter
import math
import re

import pandas as pd
import torch
from torch import nn
from llm_engineering_lab.acceleration import get_accelerator

SEED = 42
torch.manual_seed(SEED)
torch.set_printoptions(precision=4, sci_mode=False)
ACCELERATOR = get_accelerator()
device = ACCELERATOR.device
torch.set_default_device(device)


def parameter_count(model):
    return sum(
        parameter.numel() for parameter in model.parameters() if parameter.requires_grad
    )


print("torch:", torch.__version__, "|", ACCELERATOR.summary())

## Sequence shape 계약

이 노트북은 기본적으로 `[B,T,D]`를 사용합니다.

- `B`: batch size
- `T`: sequence length(time/token)
- `D`: feature/embedding dimension

`Conv1d`만 `[B,C,T]`를 요구하므로 forward 안에서 `transpose(1,2)`합니다. shape 변환을 모델 경계에 모아 두면 디버깅하기 쉽습니다.

## Part A. 작은 시계열: 다음 값 예측

두 주파수가 섞인 noisy sine series를 길이 24의 window로 잘라 다음 한 값을 예측합니다. 미래 window로 validation을 구성해 시간 누수를 막습니다.

In [ ]:
# TODO 1: Part A. 작은 시계열: 다음 값 예측
# 오른쪽 정답 창의 같은 셀을 참고하되, 먼저 아래 구현을 직접 작성하세요.
# 1) 입력과 출력의 shape/dtype을 종이에 적습니다.
# 2) 최소 구현을 작성하고 Shift+Enter로 실행합니다.
# 3) 정답과 비교한 뒤 값 하나를 바꾸고 결과를 설명합니다.
raise NotImplementedError("08번 실습의 TODO 1을 완성하세요")

In [ ]:
# 가장 최근 값을 그대로 예측하는 naive baseline
naive_prediction = valid_x[:, -1, :]
naive_mse = nn.functional.mse_loss(naive_prediction, valid_y).item()
print("naive validation MSE:", naive_mse)

# 시계열 split에서는 train의 마지막 target이 valid의 첫 target보다 과거여야 합니다.
print("first validation input tail/target:", valid_x[0, -3:, 0], valid_y[0, 0])

### 1D CNN

kernel size 3은 한 층에서 인접한 세 시점을 봅니다. 여러 층을 쌓으면 receptive field가 넓어집니다. convolution은 모든 위치를 병렬로 계산할 수 있습니다.

In [ ]:
# TODO 2: 1D CNN
# 오른쪽 정답 창의 같은 셀을 참고하되, 먼저 아래 구현을 직접 작성하세요.
# 1) 입력과 출력의 shape/dtype을 종이에 적습니다.
# 2) 최소 구현을 작성하고 Shift+Enter로 실행합니다.
# 3) 정답과 비교한 뒤 값 하나를 바꾸고 결과를 설명합니다.
raise NotImplementedError("08번 실습의 TODO 2을 완성하세요")

### GRU: RNN 계열

GRU는 시점을 순서대로 읽으며 hidden state를 갱신합니다. update/reset gate가 vanilla RNN보다 긴 의존성의 gradient 흐름을 돕습니다. 마지막 hidden state를 다음 값 예측에 사용합니다.

In [ ]:
# TODO 3: GRU: RNN 계열
# 오른쪽 정답 창의 같은 셀을 참고하되, 먼저 아래 구현을 직접 작성하세요.
# 1) 입력과 출력의 shape/dtype을 종이에 적습니다.
# 2) 최소 구현을 작성하고 Shift+Enter로 실행합니다.
# 3) 정답과 비교한 뒤 값 하나를 바꾸고 결과를 설명합니다.
raise NotImplementedError("08번 실습의 TODO 3을 완성하세요")

### Scaled dot-product attention

$$\mathrm{Attention}(Q,K,V)=\mathrm{softmax}\left(\frac{QK^T}{\sqrt{d_k}}+M\right)V$$

각 query 위치가 key 위치를 얼마나 참고할지 score를 만들고 value의 가중합을 계산합니다. 순서 정보는 연산 자체에 없으므로 positional embedding을 추가합니다.

In [ ]:
torch.manual_seed(SEED)
demo_sequence = torch.randn(2, 4, 8)  # [B,T,D]
q = k = v = demo_sequence
scores = q @ k.transpose(-2, -1) / math.sqrt(q.shape[-1])  # [B,T,T]

causal_mask = torch.triu(torch.ones(4, 4, dtype=torch.bool), diagonal=1)
masked_scores = scores.masked_fill(causal_mask, float("-inf"))
attention_weights = masked_scores.softmax(dim=-1)
attended = attention_weights @ v

print("scores/weights:", scores.shape, attention_weights.shape)
print("attended:", attended.shape)
print("first attention matrix:\n", attention_weights[0])
assert torch.allclose(attention_weights.sum(-1), torch.ones(2, 4))
assert torch.equal(attention_weights[:, 0, 1:], torch.zeros(2, 3))

In [ ]:
# TODO 4: Scaled dot-product attention
# 오른쪽 정답 창의 같은 셀을 참고하되, 먼저 아래 구현을 직접 작성하세요.
# 1) 입력과 출력의 shape/dtype을 종이에 적습니다.
# 2) 최소 구현을 작성하고 Shift+Enter로 실행합니다.
# 3) 정답과 비교한 뒤 값 하나를 바꾸고 결과를 설명합니다.
raise NotImplementedError("08번 실습의 TODO 4을 완성하세요")

In [ ]:
def fit_regressor(model, train_x, train_y, epochs=80, learning_rate=0.01):
    torch.manual_seed(SEED)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        loss = nn.functional.mse_loss(model(train_x), train_y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        if epoch in {1, 20, 40, 60, 80}:
            history.append((epoch, loss.item()))
    model.eval()
    with torch.inference_mode():
        validation_mse = nn.functional.mse_loss(model(valid_x), valid_y).item()
    return history, validation_mse

In [ ]:
torch.manual_seed(SEED)
time_models = {
    "CNN": CNNForecaster(),
    "GRU": GRUForecaster(),
    "Attention": AttentionForecaster(),
}
time_results = []

for name, candidate in time_models.items():
    history, validation_mse = fit_regressor(candidate, train_x, train_y)
    time_results.append(
        {
            "model": name,
            "parameters": parameter_count(candidate),
            "first_train_mse": history[0][1],
            "last_train_mse": history[-1][1],
            "validation_mse": validation_mse,
        }
    )
    print(name, history, "valid=", round(validation_mse, 5))

In [ ]:
time_comparison = pd.DataFrame(time_results).sort_values("validation_mse")
baseline_row = pd.DataFrame(
    [
        {
            "model": "Naive-last-value",
            "parameters": 0,
            "first_train_mse": float("nan"),
            "last_train_mse": float("nan"),
            "validation_mse": naive_mse,
        }
    ]
)
pd.concat([time_comparison, baseline_row], ignore_index=True).round(6)

In [ ]:
# padding mask와 causal mask는 목적이 다릅니다.
token_mask = torch.tensor([[1, 1, 1, 1], [1, 1, 0, 0]], dtype=torch.bool)
key_padding_mask = ~token_mask  # MultiheadAttention은 무시할 위치가 True
causal_mask = torch.triu(torch.ones(4, 4, dtype=torch.bool), diagonal=1)

print("token mask [B,T]:\n", token_mask)
print("key padding mask [B,T]:\n", key_padding_mask)
print("causal mask [T,T]:\n", causal_mask)
assert key_padding_mask.shape == (2, 4) and causal_mask.shape == (4, 4)

## Part B. 작은 텍스트 분류

같은 세 계열을 `결제 / 계정 / 배송` 문장에 적용합니다. 데이터가 매우 작으므로 일반화 점수 비교가 아니라 **입력 shape와 모델 연결을 검증하는 overfit smoke test**입니다.

In [ ]:
examples = [
    ("카드가 두 번 결제됐어요", "billing"),
    ("환불 금액이 아직 안 왔어요", "billing"),
    ("청구서 금액을 확인해 주세요", "billing"),
    ("결제 승인이 계속 실패합니다", "billing"),
    ("중복 결제를 취소하고 싶어요", "billing"),
    ("영수증을 다시 보내 주세요", "billing"),
    ("비밀번호 재설정이 안 돼요", "account"),
    ("로그인할 수 없습니다", "account"),
    ("계정이 잠겼어요", "account"),
    ("인증 코드가 오지 않아요", "account"),
    ("이메일 주소를 변경하고 싶어요", "account"),
    ("새 기기 로그인이 차단됐어요", "account"),
    ("택배가 아직 도착하지 않았어요", "delivery"),
    ("배송 조회가 멈췄어요", "delivery"),
    ("주소를 잘못 입력했습니다", "delivery"),
    ("상품이 다른 곳으로 배송됐어요", "delivery"),
    ("배송 예정일을 알려 주세요", "delivery"),
    ("분실된 택배를 찾아 주세요", "delivery"),
]
TEXT_PATTERN = re.compile(r"[가-힣]|[A-Za-z]+|\d+|[^\w\s]")
tokenize = lambda text: TEXT_PATTERN.findall(text.lower())

counts = Counter(token for text, _ in examples for token in tokenize(text))
itos = ["<pad>", "<unk>"] + sorted(counts, key=lambda token: (-counts[token], token))
stoi = {token: index for index, token in enumerate(itos)}
label_names = sorted({label for _, label in examples})
label_to_id = {label: index for index, label in enumerate(label_names)}

print("vocab/classes:", len(itos), label_names)
print("tokens:", tokenize(examples[0][0]))

In [ ]:
encoded_texts = [
    torch.tensor([stoi.get(token, 1) for token in tokenize(text)], dtype=torch.long)
    for text, _ in examples
]
text_ids = nn.utils.rnn.pad_sequence(encoded_texts, batch_first=True, padding_value=0)
text_mask = text_ids.ne(0)
text_labels = torch.tensor([label_to_id[label] for _, label in examples])

print("ids/mask/labels:", text_ids.shape, text_mask.shape, text_labels.shape)
print("first ids:", text_ids[0])
print("first mask:", text_mask[0])
assert text_ids.shape == text_mask.shape

In [ ]:
class CNNTextClassifier(nn.Module):
    def __init__(self, vocab_size, num_classes, model_dim=24):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, model_dim, padding_idx=0)
        self.convolution = nn.Conv1d(model_dim, 32, kernel_size=3, padding=1)
        self.output = nn.Linear(32, num_classes)

    def forward(self, ids, mask):
        embedded = self.embedding(ids).transpose(1, 2)  # [B,D,T]
        hidden = torch.relu(self.convolution(embedded))
        hidden = hidden.masked_fill(~mask.unsqueeze(1), -1e4)
        pooled = hidden.max(dim=-1).values
        return self.output(pooled)


class GRUTextClassifier(nn.Module):
    def __init__(self, vocab_size, num_classes, model_dim=24):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, model_dim, padding_idx=0)
        self.gru = nn.GRU(model_dim, 32, batch_first=True)
        self.output = nn.Linear(32, num_classes)

    def forward(self, ids, mask):
        sequence, _ = self.gru(self.embedding(ids))
        lengths = mask.sum(-1).clamp_min(1)
        last = sequence[torch.arange(len(ids)), lengths - 1]
        return self.output(last)

In [ ]:
class AttentionTextClassifier(nn.Module):
    def __init__(self, vocab_size, num_classes, max_length, model_dim=24, num_heads=4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, model_dim, padding_idx=0)
        self.position = nn.Parameter(torch.randn(1, max_length, model_dim) * 0.02)
        self.attention = nn.MultiheadAttention(model_dim, num_heads, batch_first=True)
        self.norm = nn.LayerNorm(model_dim)
        self.output = nn.Linear(model_dim, num_classes)

    def forward(self, ids, mask):
        hidden = self.embedding(ids) + self.position[:, : ids.shape[1]]
        attended, _ = self.attention(
            hidden, hidden, hidden, key_padding_mask=~mask, need_weights=False
        )
        hidden = self.norm(hidden + attended)
        weights = mask.unsqueeze(-1).to(hidden.dtype)
        pooled = (hidden * weights).sum(1) / weights.sum(1).clamp_min(1.0)
        return self.output(pooled)

In [ ]:
torch.manual_seed(SEED)
text_models = {
    "CNN": CNNTextClassifier(len(itos), len(label_names)),
    "GRU": GRUTextClassifier(len(itos), len(label_names)),
    "Attention": AttentionTextClassifier(
        len(itos), len(label_names), max_length=text_ids.shape[1]
    ),
}

architecture_rows = []
for name, candidate in text_models.items():
    logits = candidate(text_ids, text_mask)
    architecture_rows.append(
        {
            "model": name,
            "logits_shape": tuple(logits.shape),
            "parameters": parameter_count(candidate),
        }
    )
    assert logits.shape == (len(examples), len(label_names))
pd.DataFrame(architecture_rows)

In [ ]:
def overfit_text_model(model, steps=100):
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.02)
    losses = []
    for step in range(steps):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        logits = model(text_ids, text_mask)
        loss = nn.functional.cross_entropy(logits, text_labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        if step in {0, 24, 49, 99}:
            losses.append((step + 1, loss.item()))
    model.eval()
    with torch.inference_mode():
        predictions = model(text_ids, text_mask).argmax(-1)
    return losses, predictions.eq(text_labels).float().mean().item()


text_results = []
for name, candidate in text_models.items():
    losses, accuracy = overfit_text_model(candidate)
    text_results.append(
        {
            "model": name,
            "parameters": parameter_count(candidate),
            "final_loss": losses[-1][1],
            "train_accuracy": accuracy,
        }
    )
    print(name, losses, "accuracy=", accuracy)

In [ ]:
comparison = pd.DataFrame(text_results).sort_values("parameters")
print(comparison.round(5))

# 학습 샘플 예측을 확인합니다. 이 점수는 validation 성능이 아닙니다.
selected_model = text_models["Attention"]
with torch.inference_mode():
    probabilities = selected_model(text_ids, text_mask).softmax(-1)
    predicted_ids = probabilities.argmax(-1)

prediction_table = pd.DataFrame(
    {
        "text": [text for text, _ in examples],
        "truth": [label for _, label in examples],
        "prediction": [label_names[index] for index in predicted_ids.tolist()],
        "confidence": probabilities.max(-1).values.tolist(),
    }
)
prediction_table

## 모델 선택 가이드

- 짧은 local motif가 중요하고 latency가 빡빡하면 1D CNN부터 기준 모델을 만듭니다.
- streaming처럼 입력이 순차 도착하고 state를 유지해야 하면 GRU/RNN이 자연스럽습니다.
- 멀리 떨어진 위치의 상호작용과 대규모 병렬 학습이 중요하면 attention이 유리합니다.
- sequence가 매우 길면 vanilla attention의 $T^2$ 메모리를 반드시 측정합니다.
- 실제 선택은 이론만으로 하지 않고 동일 split에서 품질, latency, peak memory, parameter 수를 함께 측정합니다.

이 노트북의 작은 train accuracy는 모델 용량 점검용이며 일반화 성능을 뜻하지 않습니다.

## 따라 치기 TODO + 확장 과제

1. 시계열 window를 `8, 24, 64`로 바꾸고 세 모델의 MSE·학습 시간을 비교하세요.
2. CNN에 dilation을 추가하고 receptive field를 직접 계산하세요.
3. GRU를 `nn.RNN`, `nn.LSTM`으로 바꾸어 parameter 수와 수렴 속도를 비교하세요.
4. attention의 positional embedding을 제거하고 성능 변화를 설명하세요.
5. text 데이터를 group-safe train/validation으로 늘리고 macro F1을 측정하세요.
6. `torch.utils.benchmark`로 sequence length에 따른 forward latency를 측정하세요.
7. attention weight를 반환해 특정 문장이 어느 글자에 집중했는지 표로 만드세요.
8. causal mask가 필요한 생성 모델과 필요 없는 문장 분류 모델의 차이를 예제로 설명하세요.

핵심은 모델 이름이 아니라 **입력/출력 shape, mask 의미, 정보가 섞이는 경로, 계산 비용을 설명할 수 있는가**입니다.